# Section 3: Robustness, Where You Go Beyond (28 pts)

ECON 5200 Midterm Project · Replication Study

**AI is allowed in this notebook**, as a documented co-pilot under P.R.I.M.E. File every exchange, in full, in [`ai-appendix.md`](../ai-appendix.md) under the exercise it belongs to. If you cannot explain a line, you cannot submit it.

At least four exercises the original did not run. Choose from modern estimators, a specification curve, sensitivity to unobserved confounding, inference (clustering at the level of treatment assignment, a wild cluster bootstrap if clusters are few), bandwidth or instrument strength, and placebo tests, or justify an alternative. For each one: what you did, what you found, and what it implies about the original claim.

## Setup

In [ ]:
!pip install rdrobust

import sys
from pathlib import Path
from rdrobust import rdrobust
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# The repository root, whether this runs from notebooks/ (Jupyter) or from the root (reproduce.py).
ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(exist_ok=True)

from download_data import MY_PAPER, PAPERS, fetch

## Data

In [ ]:
# Set MY_PAPER once, in download_data.py. fetch() downloads into data/ only if a file is missing.
if MY_PAPER:
    paths = fetch(MY_PAPER)
    print(PAPERS[MY_PAPER]["paper"])
    dwi = pd.read_stata(paths[0])

else:
    print("TODO: set MY_PAPER in download_data.py (castle, card, lee, dw, hansen or abortion).")

downloading https://raw.githubusercontent.com/scunning1975/mixtape/master/hansen_dwi.dta
  data/hansen_dwi.dta  9.5 MB
Hansen (2015), punishment and deterrence in drunk driving


In [ ]:
CUTOFF = 0.08
dwi["bac_min"] = dwi["low_score"].astype(float) / 1000
RUNNING = "bac_min"
H = 0.05
CONTROLS = ["male", "white", "aged", "acc"]
def rd(cutoff, h=None, kernel="triangular", controls=None, data=None, outcome="recidivism", donut=0.0):
    data = dwi if data is None else data
    d = data[(data[RUNNING] - cutoff).abs() >= donut]
    covs = d[controls] if controls else None
    kw = {"h": h} if h else {}
    r = rdrobust(y=d[outcome], x=d[RUNNING], c=cutoff, p=1, kernel=kernel, covs=covs, **kw)
    return {"jump": r.coef.iloc[0, 0], "se": r.se.iloc[0, 0], "p_robust": r.pv.iloc[2, 0],
            "h": r.bws.iloc[0, 0], "n": int(r.N_h[0] + r.N_h[1])}


def show(label, r):
    print(f"{label:<34} {r['jump']:+.4f} (SE {r['se']:.4f}, robust p {r['p_robust']:.3f}), h = {r['h']:.3f}, n = {r['n']:,}")


## The four exercises

### Exercise 1: Change the Bandwidths

**What I did.** I still used the 0.08 cutoff but ran different bandwidths that the paper did not use.

In [ ]:
from rdrobust import rdrobust

r_025 = rd(0.08, h=0.025, kernel="uniform", controls=CONTROLS)
show("h = 0.025", r_025)
r_same = rd(0.08, h=0.05, kernel="uniform", controls=CONTROLS)
show("h = 0.05", r_same)
r_1 = rd(0.08, h=0.1, kernel="uniform", controls=CONTROLS)
show("h = 0.10", r_1)
r_15 = rd(0.08, h=0.15, kernel="uniform", controls=CONTROLS)
show("h = 0.15", r_15)

Mass points detected in the running variable.
h = 0.025                          -0.0196 (SE 0.0057, robust p 0.131), h = 0.025, n = 50,016
Mass points detected in the running variable.
h = 0.05                           -0.0222 (SE 0.0043, robust p 0.035), h = 0.050, n = 93,899
Mass points detected in the running variable.
h = 0.10                           -0.0207 (SE 0.0034, robust p 0.000), h = 0.100, n = 169,916
Mass points detected in the running variable.
h = 0.15                           -0.0210 (SE 0.0033, robust p 0.000), h = 0.150, n = 203,342


**What I found.** I found the jumps stay failrly close at -0.0196, -0.0207, and -0.0210. These are all close to the papers value of -0.021.

**What it implies about the original claim.** This implies that the Hansen was correct in saying that the effect does not depend on the bandwidth choice because the estimate was very similar for each bandwidth i tested. Looking at the smallest bandwidth of 0.025, the robust p-value is 0.131 which means the evidence is weak and not signifigant. At H=0.1, there is not a clear estiamte because it includes a BAC of 0.15 which means the drivers would face a different punishment.

### Exercise 2: Placebo Cutoffs

**What I did.** I tested the data at cutoffs 0.05 and 0.11 where we expect the law to do nothing. I used the h=0.025 bandwidth so the range would not hit the cutoffs

In [ ]:
r_005 = rd(0.05, h=0.025, kernel="uniform", controls=CONTROLS)
show("placebo 0.05", r_005)

r_011 = rd(0.11, h=0.025, kernel="uniform", controls=CONTROLS)
show("placebo 0.11", r_011)

Mass points detected in the running variable.
placebo 0.05                       -0.0201 (SE 0.0129, robust p 0.131), h = 0.025, n = 16,417
Mass points detected in the running variable.
placebo 0.11                       +0.0005 (SE 0.0044, robust p 0.669), h = 0.025, n = 75,303


**What I found.** The 0.05 showed -0.0201 with p=0.131 and 0.11 found a 0.0005 jump with p=0.67

**What it implies about the original claim.** These results make senese. At the 0.05 the results showed a jump similar to what the paper measured (-0.021) however, with p=0.131, this result is not signifigant. This is likely because it is a much lower BAC so a smaller sample so there was likely more noise. Looking at 0.11, the jump was zero which is expected.

### Exercise 3: Different Regression Discontinuity method

**What I did.** At 0.08 and 0.15 I used rdrobust, which picks specific bandwidths from the data and reports robust p-values. However, Rdrobust uses a triangle kernel which differs from the rectangle used in the paper.

In [ ]:
r1 = rd(0.08, controls=CONTROLS)
show("cutoff 0.08", r1)

r2 = rd(0.15, controls=CONTROLS)
show("cutoff 0.15", r2)

Mass points detected in the running variable.
cutoff 0.08                        -0.0173 (SE 0.0060, robust p 0.024), h = 0.027, n = 53,565
Mass points detected in the running variable.
cutoff 0.15                        -0.0075 (SE 0.0048, robust p 0.203), h = 0.029, n = 84,225


**What I found.** When using rdrobust, at 0.08 I found the jump to be -0.0173 with p=0.024 and at 0.15 the jump is -0.0075 with p=.203.

**What it implies about the original claim.** The effect at 0.08 still exists but it is a bit smaller coming in at -0.0173 (compared to Hansen's -0.021). Because the kernel is different in this exercise compared to the paper and uses a different bandwidth, you cannot conclude much from this, however, the jumps are somewhat close together. The jump at 0.15 is much smaller (-0.0075) and is not statistically different from zero. The paper's story is that harsher punishment deters drunk driving, which predicts a response at both cutoffs. So the evidence for the claim is stronger at 0.08 than at 0.15.

### Exercise 4: Donut for Regression Discontinuity

**What I did.** Instead of the rectangular shape, I used a donut around the 0.08 cutoff. This removed all the drivers whose BAC was within 0.002 of 0.08 then ran the original replication again keeping everything else the same.

In [ ]:
import numpy as np
import pandas as pd
from rdrobust import rdrobust

MY_SPEC = {"cutoff": 0.08, "h": 0.05, "kernel": "uniform", "donut": 0.002, "sample": None}

# --- adapt these names to your notebook ---
RV       = "bac_min"            # your running variable (low_score/1000)
Y        = "recidivism"         # your outcome column
CONTROLS = ["male", "white", "aged", "acc"]
CLUSTER  = RV                   # you cluster by the running variable's values

def run_spec(df, spec):
    d = df.query(spec["sample"]) if spec["sample"] else df
    d = d.dropna(subset=[Y, RV, "year"] + CONTROLS)
    d = d[(d[RV] - spec["cutoff"]).abs() >= spec["donut"]]      # the donut
    yr = pd.get_dummies(d["year"], prefix="yr", drop_first=True).astype(float)
    covs = pd.concat([d[CONTROLS], yr], axis=1)
    kw = dict(c=spec["cutoff"], p=1, kernel=spec["kernel"], covs=covs, cluster=d[CLUSTER])
    if spec["h"] is not None:
        kw["h"] = spec["h"]
    res = rdrobust(d[Y], d[RV], **kw)
    return res, len(d)

# 1) Heaping diagnostic: are readings piling up near the cutoff?
win = dwi[(dwi[RV] - 0.08).abs() <= 0.01]
counts = win[RV].round(3).value_counts().sort_index()
print(counts)

# 2) Donut sweep, with MY_SPEC as the main result (donut = 0 reproduces the baseline)
rows = []
for dn in [0, 0.001, MY_SPEC["donut"], 0.003]:
    spec = {**MY_SPEC, "donut": dn}
    res, n = run_spec(dwi, spec)
    rows.append({
        "donut": dn,
        "estimate": res.coef.iloc[0, 0],   # conventional
        "se": res.se.iloc[0, 0],
        "p": res.pv.iloc[0, 0],
        "N used": n,
    })
donut_table = pd.DataFrame(rows).round(4)
donut_table

bac_min
0.070     772
0.071     757
0.072     829
0.073     810
0.074     840
0.075     872
0.076     887
0.077     922
0.078     975
0.079     949
0.080     998
0.081    1075
0.082    1163
0.083    1167
0.084    1095
0.085    1144
0.086    1168
0.087    1237
0.088    1254
0.089    1315
0.090    1292
Name: count, dtype: int64
Mass points detected in the running variable.
Mass points detected in the running variable.
Mass points detected in the running variable.
Mass points detected in the running variable.


,donut,estimate,se,p,N used
0,0.000,-0.0222,0.0042,0.0,214558
1,0.001,-0.0222,0.0042,0.0,213560
2,0.002,-0.0231,0.0045,0.0,211536
3,0.003,-0.0240,0.0049,0.0,209398


**What I found.** I found that with the main donut of 0.002 the jump was -0.0231 (SE 0.0045). This dropped about 3,000 observations. Using a larger donut at 0.003 the jump moved out to -0.0240. For both, the result was signifigant and the standard error grew a littl bit.

**What it implies about the original claim.** After dropping the drivers within 0.002 of the cutoff, the effect at 0.08 still exists. The jump is measured at -0.0231. This can be directly compared to my baseline (-0.0222) because it uses the same rectangular kernel and bandwidth as my replication. The difference is about 0.001 which is smaller than the standard error (0.0045), meaning that I cannot conclude the donut actually changed anything. Hansen says that harsher punishment at 0.08 deters repeat drunk driving, which predicts a drop in recidivism right at the cutoff. If certain errors were creating this pattern, removing them with the donut should have shrunk it, and it did not. So the evidence for the claim holds up at 0.08. The donut only tests problems very close to the cutoff, though, so it cannot rule out drivers sorting across the threshold from farther away.